In [13]:
!hostname

2271.39s - pydevd: Sending message related to process being replaced timed-out after 5 seconds


ithor.ifl.campar.in.tum.de


In [ ]:
import pandas as pd
df = pd.read_csv("phenotype.hpoa", sep="\t", comment="#", low_memory=False)
# print(df.head())

print(df.columns)
print(df.shape)

rare_resources = ["OMIM", "ORPHA", "ORPHANET"]

# we keep just the phenotypes that are associated with OMIM/ORPHANET -> so they are just rare diseases
df = df[df["database_id"].str.split(":").str[0].isin(rare_resources)]  
print(df.shape)

reference_pmid = df["reference"].str.startswith("PMID:")

pmid_list = (
    df['reference']
      .str.split(';')
      .explode()
      .str.strip()
      .loc[lambda s: s.str.startswith('PMID:')]
      .str.split(':', n=1).str[1]
      .drop_duplicates()
      .tolist()
)
print(len(pmid_list))
print(pmid_list[:10])




Index(['database_id', 'disease_name', 'qualifier', 'hpo_id', 'reference',
       'evidence', 'onset', 'frequency', 'sex', 'modifier', 'aspect',
       'biocuration'],
      dtype='object')
(272726, 12)
(272430, 12)
8924
['31675180', '2766660', '24947683', '26853951', '3931219', '33743206', '15890322', '19890111', '21519361', '7740448']


In [12]:
import pandas as pd


df = pd.read_csv("Gene-RD-Provenance_V2.1.txt", sep="\t", dtype=str).fillna("")


print(df.columns)
print(df.shape)
print(sum(df["PMID Gene-disease"] == df["PMID Disease"]))


# Concatenate the two columns, drop NA and duplicates, and get unique PMIDs
pmids = pd.concat([df["PMID Gene-disease"], df["PMID Disease"]])
#pmids = pmids[pmids != ""] # Remove empty strings if any
pmids = pmids.dropna().drop_duplicates().tolist()
print(len(pmids))


Index(['ENSID', 'HGNC', 'PMID Gene-disease', 'Disease OMIM ID', 'Disease name',
       'PMID Disease'],
      dtype='object')
(4565, 6)
1278
6655


In [1]:
import sys, site, subprocess, json
print(sys.executable)
print(site.getsitepackages())
import requests, pandas as pd
print("OK:", requests.__version__, pd.__version__)


/home/guests/andreea_magureanu/.conda/envs/rare_dis/bin/python
['/home/guests/andreea_magureanu/.conda/envs/rare_dis/lib/python3.11/site-packages']


OK: 2.32.4 2.3.1


In [ ]:
# --------- setup ---------
from Bio import Entrez
import requests, time, xml.etree.ElementTree as ET
from typing import Iterator, Dict, List, Tuple

# identify yourself to NCBI
Entrez.email = "amagureanuandreea13@gmail.com"           # <-- put your email
Entrez.api_key = "6755b8486d7b82533be134b4aad7814e2908"       # <-- optional but recommended

# --------- discovery (optional): query -> PMIDs ---------
def esearch_pmids(query: str, retmax: int = 100_000) -> List[str]:
    """Get PMIDs matching a PubMed query."""
    h = Entrez.esearch(db="pubmed", term=query, retmax=retmax)
    rec = Entrez.read(h)
    return rec["IdList"]

# --------- register IDs with EPost (Biopython) ---------
def epost_pmids(pmids: List[str]) -> Tuple[str, str]:
    """Upload many PMIDs once; returns (WebEnv, QueryKey)."""
    post = Entrez.epost(db="pubmed", id=",".join(pmids))
    rec = Entrez.read(post)
    return rec["WebEnv"], rec["QueryKey"]

# --------- minimal XML parsing helpers ---------
def _extract_text(elem) -> str:
    return "".join(elem.itertext()).strip() if elem is not None else ""

def parse_pubmed_xml(xml_text: str) -> List[Dict[str, str]]:
    """Turn an EFetch XML page into a list of {pmid,title,abstract} dicts."""
    root = ET.fromstring(xml_text)
    out = []
    for art in root.findall(".//PubmedArticle"):
        pmid = art.findtext(".//PMID") or ""
        title = _extract_text(art.find(".//ArticleTitle"))
        parts = []
        for at in art.findall(".//Abstract/AbstractText"):
            label = at.get("Label")
            text = _extract_text(at)
            parts.append(f"{label}: {text}" if label else text)
        abstract = "\n\n".join([p for p in parts if p])
        out.append({"pmid": pmid, "title": title, "abstract": abstract})
    return out

# --------- fetch via WebEnv/QueryKey (requests + paging) ---------
def efetch_iter(webenv: str, query_key: str,
                retmax: int = 200, pause: float = 0.34,
                api_key: str | None = Entrez.api_key) -> Iterator[Dict[str, str]]:
    """
    Iterate over all results registered by EPost using WebEnv/QueryKey.
    Yields dicts with pmid/title/abstract.
    """
    base = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi"
    retstart = 0
    session = requests.Session()

    while True:
        params = {
            "db": "pubmed",
            "retmode": "xml",
            "retstart": retstart,  # start index
            "retmax": retmax,      # page size 
            "query_key": query_key,
            "WebEnv": webenv,
            "tool": "andreea-rare-disease-pipeline",
            "email": Entrez.email,
        }
        if api_key:
            params["api_key"] = api_key

        # simple retry on rate limit
        for attempt in range(3):
            r = session.get(base, params=params, timeout=60)
            if r.status_code == 429 and attempt < 2:
                time.sleep(2 ** attempt)
                continue
            r.raise_for_status()
            break

        batch = parse_pubmed_xml(r.text)
        if not batch:
            break  # no more records
        for rec in batch:
            yield rec

        retstart += retmax
        time.sleep(pause)  # stay friendly to NCBI

# --------- convenience: glue it all together ---------
def fetch_titles_abstracts_from_pmids(pmids: List[str],
                                      page_size: int = 200) -> List[Dict[str, str]]:
    """
    EPost the list with Biopython, then EFetch+parse with our minimal parser.
    Returns a list of dicts.
    """
    webenv, qk = epost_pmids(pmids)
    out = []
    for rec in efetch_iter(webenv, qk, retmax=page_size):
        out.append(rec)
    return out

# --------- optional: build TF‑IDF ---------
def build_tfidf(records: List[Dict[str, str]],
                max_features: int = 50_000,
                ngram_range: tuple = (1, 2)):
    from sklearn.feature_extraction.text import TfidfVectorizer
    docs, ids = [], []
    for r in records:
        text = " ".join([t for t in (r["title"], r["abstract"]) if t]).strip()
        if text:
            docs.append(text)
            ids.append(r["pmid"])
    vec = TfidfVectorizer(max_features=max_features, ngram_range=ngram_range, lowercase=True)
    X = vec.fit_transform(docs)  # sparse matrix
    return ids, X, vec

# --------- example usage ---------
if __name__ == "__main__":
    # Option A: you already have PMIDs (recommended for your pipeline)
    pmids = ["31115799"]  # replace with your list
    records = fetch_titles_abstracts_from_pmids(pmids)

    # Option B (optional): discover PMIDs first via a query
    # q_pmids = esearch_pmids('("Duchenne muscular dystrophy"[Title/Abstract]) AND 2020:2025[dp]')
    # records = fetch_titles_abstracts_from_pmids(q_pmids)

    # Now build TF‑IDF
    ids, X, vectorizer = build_tfidf(records)
    print(f"TF-IDF matrix: {X.shape}, first 5 PMIDs: {ids[:5]}")


HTTPError: 400 Client Error: Bad Request for url: https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi?db=pubmed&retmode=xml&retstart=200&retmax=200&query_key=1&WebEnv=MCID_68a0bc1c6a6018db53063916&tool=andreea-rare-disease-pipeline&email=amagureanuandreea13%40gmail.com&api_key=6755b8486d7b82533be134b4aad7814e2908

In [3]:


from __future__ import annotations

from typing import List, Dict, Tuple, Any

import numpy as np
from sklearn.pipeline import make_pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import precision_score, recall_score, f1_score
import matplotlib.pyplot as plt


class RareDiseaseClassifier:
   

    def __init__(self, *, max_features: int = 50_000, ngram_range: Tuple[int, int] = (1, 2), seed: int = 42) -> None:
        self.max_features = max_features
        self.ngram_range = ngram_range
        self.seed = seed

        self.model = make_pipeline(
            TfidfVectorizer(
                lowercase=True,
                max_features=self.max_features,
                ngram_range=self.ngram_range,
                min_df=2,
            ),
            LogisticRegression(
                max_iter=1000,
                class_weight="balanced",
                solver="lbfgs",
                n_jobs=-1,
            ),
        )

    @staticmethod
    def _join_title_abstract(record: Dict[str, Any]) -> str:
    
        title = record.get("title", "").strip()
        abstract = record.get("abstract", "").strip()
        return f"{title}. {abstract}".strip()




    def prepare_data( self,
    pos_records: List[Dict[str, Any]],
    neg_records: List[Dict[str, Any]],) -> Tuple[List[str], np.ndarray, List[str]]:
   
        if not pos_records or not neg_records:
            raise ValueError("Both positive and negative lists must be non-empty.")

        X_text: List[str] = []
        y_list: List[int] = []
        pmids: List[str] = []

    # positives
        for rec in pos_records:
            X_text.append(self._join_title_abstract(rec))
            y_list.append(1)
            pmids.append(rec.get("pmid", ""))

    # negatives
        for rec in neg_records:
            X_text.append(self._join_title_abstract(rec))
            y_list.append(0)
            pmids.append(rec.get("pmid", ""))

        return X_text, np.array(y_list), pmids


    def cross_validate(
        self,
        X_text: List[str],
        y: np.ndarray,
        *,
        n_splits: int = 5,
    ) -> Dict[str, List[float]]:
        
        skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=self.seed)
        # Accumulate metrics per fold
        precision_scores: List[float] = []
        recall_scores: List[float] = []
        f1_scores: List[float] = []

        # Loop over each fold manually to allow custom metric computation
        for train_idx, test_idx in skf.split(X_text, y):
            # Partition the data
            X_train = [X_text[i] for i in train_idx]
            y_train = y[train_idx]
            X_test = [X_text[i] for i in test_idx]
            y_test = y[test_idx]

            # Fit a fresh instance of the model on this fold
            fold_model = make_pipeline(
                TfidfVectorizer(
                    lowercase=True,
                    max_features=self.max_features,
                    ngram_range=self.ngram_range,
                    min_df=2,
                ),
                LogisticRegression(
                    max_iter=1000,
                    class_weight="balanced",
                    solver="lbfgs",
                    n_jobs=-1,
                ),
            )
            fold_model.fit(X_train, y_train)
            y_pred = fold_model.predict(X_test)

            precision_scores.append(precision_score(y_test, y_pred))
            recall_scores.append(recall_score(y_test, y_pred))
            f1_scores.append(f1_score(y_test, y_pred))

        scores = {
            "precision": precision_scores,
            "recall": recall_scores,
            "f1": f1_scores,
        }
        return scores

    def train(self, X_text: List[str], y: np.ndarray) -> None:
        self.model.fit(X_text, y)

    def evaluate(self, X_text: List[str], y_true: np.ndarray) -> Dict[str, float]:
        """Evaluate the trained classifier on a held-out set.
        """
        
        
        if not hasattr(self.model, "predict"):
            raise ValueError("The model must be trained before evaluation.")
        y_pred = self.model.predict(X_text)
        return {
            "precision": precision_score(y_true, y_pred),
            "recall": recall_score(y_true, y_pred),
            "f1": f1_score(y_true, y_pred),
        }

    def plot_cv_results(self, scores: Dict[str, List[float]]) -> None:
        """
        Plot cross–validation metrics as simple bar charts.
        
        
        Parameters
        ----------
        
        scores : dict 
            Dictionary containing lists of per-fold metric values.  It
            must contain the keys ``precision``, ``recall`` and ``f1``.
        """
        
        metrics = ["precision", "recall", "f1"]
        for metric in metrics:
            if metric not in scores:
                raise KeyError(f"Missing metric '{metric}' in scores dictionary.")

        # Determine the number of folds 
        n_folds = len(scores["precision"])
        fold_indices = list(range(1, n_folds + 1))

        
        for metric in metrics:
            plt.figure()
            values = scores[metric]
            plt.bar(fold_indices, values)
            plt.xlabel("Fold")
            plt.ylabel(metric.capitalize())
            plt.title(f"Cross–validation {metric} per fold")
            plt.xticks(fold_indices)
            plt.ylim(0, 1)
            plt.tight_layout()
       


__all__ = ["RareDiseaseClassifier"]

In [12]:
TOOL = "andreea-rare-disease-pipeline"
EMAIL = "magureanuandreea13@gmail.com"
BASE = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils"
API_KEY = "6755b8486d7b82533be134b4aad7814e2908" 

def fetch_titles_abstracts_chunked(
    pmids: list[str],
    *,
    chunk_size: int = 200,
    pause: float = 0.34,
    retries: int = 3,
) -> list[dict[str, str]]:
    """
    Fetch PubMed titles/abstracts for a large PMID list by batching into <=chunk_size chunks.
    Uses POST /efetch.fcgi with 'id' parameter (no WebEnv/QueryKey).
    Returns a list of dicts: {"pmid", "title", "abstract"}; deduplicated by PMID.

    Args:
        pmids: list of PMID strings (can be messy; will be cleaned to digits).
        chunk_size: max PMIDs per efetch call (NCBI recommends <=200).
        pause: polite sleep between calls (seconds). (~0.34s ~ 3 req/s)
        retries: per‑chunk retry attempts on transient HTTP errors.

    Notes:
        - Requires parse_pubmed_xml(xml_text) to be defined (you already have it).
        - Respects API_KEY if set.
    """
    # Clean PMIDs to pure digits & dedup while preserving order
    seen = set()
    cleaned: list[str] = []
    for p in pmids:
        p = (p or "").strip()
        if p.isdigit() and p not in seen:
            seen.add(p)
            cleaned.append(p)

    if not cleaned:
        return []

    out: list[dict[str, str]] = []
    seen_pmids: set[str] = set()
    s = requests.Session()

    for i in range(0, len(cleaned), chunk_size):
        batch = cleaned[i : i + chunk_size]
        if not batch:
            continue

        data = {
            "db": "pubmed",
            "retmode": "xml",
            "id": ",".join(batch),
            "tool": TOOL,
            "email": EMAIL,
        }
        if API_KEY:
            data["api_key"] = API_KEY

        attempt = 0
        while True:
            try:
                r = s.post(f"{BASE}/efetch.fcgi", data=data, timeout=120)
                r.raise_for_status()
                xml = r.text
                recs = parse_pubmed_xml(xml)
                # Dedup by PMID (efetch can echo duplicates)
                for rec in recs:
                    pmid = rec.get("pmid", "")
                    if pmid and pmid not in seen_pmids:
                        seen_pmids.add(pmid)
                        out.append(rec)
                break  # success, go to next chunk
            except requests.RequestException as e:
                attempt += 1
                if attempt > retries:
                    # log and continue with next chunk
                    # (if you prefer hard‑fail, raise here)
                    print(f"[efetch] failed chunk {i//chunk_size+1} after {retries} retries: {e}")
                    break
                time.sleep(min(2**attempt, 10))  # simple backoff

        time.sleep(pause)

    return out

import pmids_extraction
from PubMed import parse_pubmed_xml
import requests, time

ids = pmids_extraction.extract_pmid_from_RD()
print(len(ids)) 
records = fetch_titles_abstracts_chunked(ids[:800])

6654


In [11]:
print(len(records))

400


In [7]:
import json
pos_path = "data/positive_records.json"
def load_json(path):
    with open(path) as f:
        return json.load(f)
    
records  = load_json(pos_path)
i= 0
for r in records:
    if r["pmid"]:
        i+=1
print(i)


396


In [3]:
import pmids_extraction
import PubMed

ids = pmids_extraction.positive_examples()
records = PubMed.fetch_titles_abstracts_chunked(ids)
print(len(records))

11875


In [4]:
print(len(ids))

12600


In [2]:
import pmids_extraction
import PubMed
pos_pmids = pmids_extraction.positive_examples()
print(f"# positive PMIDs: {len(pos_pmids)}")

    
new_pos_records = PubMed.fetch_titles_abstracts_chunked(pos_pmids, chunk_size=400)

print(len(new_pos_records))

# positive PMIDs: 12600
11875


In [4]:
import os
DATA_DIR = "data"
print("CWD:", os.getcwd())
print("DATA_DIR:", DATA_DIR)
print("Files in DATA_DIR:", os.listdir(DATA_DIR))


CWD: /home/guests/andreea_magureanu/projects/rare_disease_pipeline/paper_filter
DATA_DIR: data
Files in DATA_DIR: ['run_1.json', 'papers.db', 'rare_disease_classifier_1.pkl']


In [5]:
import PubMed
n_neg = 12600
neg_records = PubMed.sample_titles_abstracts_by_period(
            date_from="2000",
            date_to="2020",
            n_papers=n_neg,
            extra_query="",
        )
print(len(neg_records))


6806


In [4]:
model_path = "/home/guests/andreea_magureanu/projects/rare_disease_pipeline/paper_filter/data/rare_disease_classifier_1.pkl"
import pickle
with open(model_path, "rb") as f:
    model = pickle.load(f)

print(type(model))            # shows the Python class
print(dir(model)[:50]) 

<class 'filter_classifier.RareDiseaseClassifier'>
['__class__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattribute__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__lt__', '__module__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__sizeof__', '__str__', '__subclasshook__', '__weakref__', '_join_title_abstract', 'cross_validate', 'evaluate', 'max_features', 'model', 'ngram_range', 'plot_cv_results', 'prepare_data', 'seed', 'train']


In [ ]:
# def _esearch_all_pmids(term: str, retmax: int = 10000, pause: float = 0.34) -> List[str]:
    
#     params = {"db":"pubmed","term":term,"retmode":"xml","retmax":0, "tool":TOOL, "email":EMAIL}
    
#     if API_KEY: params["api_key"] = API_KEY
    
#     r = requests.get(f"{BASE}/esearch.fcgi", params=params, timeout=60)
#     r.raise_for_status()
    
#     root = ET.fromstring(r.text)
    
#     total = int(root.findtext("Count", default="0"))
    
#     pmids: List[str] = []
    
#     for start in range(0, total, retmax):
#         params.update({"retstart": start, "retmax": retmax})
#         r = requests.get(f"{BASE}/esearch.fcgi", params=params, timeout=90)
#         r.raise_for_status()
#         root = ET.fromstring(r.text)
#         ids = [e.text for e in root.findall("./IdList/Id") if e is not None and e.text]
#         pmids.extend(ids)
#         time.sleep(pause)
        
#     return pmids


def build_term(date_from: str, date_to: str, extra_query: str = "") -> str:
    base = f'({date_from}:{date_to}[dp] AND hasabstract[text])'
    return base if not extra_query else f'{base} AND ({extra_query})'


import xml.etree.ElementTree as ET, requests
term = "2020/1/1:2020/1/2[dp] AND hasabstract[text]"
TOOL = "andreea-rare-disease-pipeline"
EMAIL = "magureanuandreea13@gmail.com"
BASE = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils"
API_KEY = "6755b8486d7b82533be134b4aad7814e2908" 

params = {"db":"pubmed","term":term,"retmode":"xml","retmax":0, "tool":TOOL, "email":EMAIL}
    
if API_KEY: params["api_key"] = API_KEY
    
r = requests.get(f"{BASE}/esearch.fcgi", params=params, timeout=60)
r.raise_for_status()
print(r.text)
root = ET.fromstring(r.text)
# root is just an element  
print(root) 
total = int(root.findtext("Count", default="0"))
print(total)

 

<?xml version="1.0" encoding="UTF-8" ?>
<!DOCTYPE eSearchResult PUBLIC "-//NLM//DTD esearch 20060628//EN" "https://eutils.ncbi.nlm.nih.gov/eutils/dtd/20060628/esearch.dtd">
<eSearchResult><Count>218544</Count><RetMax>0</RetMax><RetStart>0</RetStart><IdList/><TranslationSet/><QueryTranslation>2020/01/01:2020/01/02[Date - Publication] AND "hasabstract"[Text Word]</QueryTranslation></eSearchResult>

<Element 'eSearchResult' at 0x7f3860b66a70>
